In [ ]:
!pip install pyhealth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 622.3/622.3 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.4/205.4 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.4/426.4 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 M

In [ ]:
import tempfile

from pyhealth.datasets import MIMIC3Dataset
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.models import Transformer
from pyhealth.tasks import DrugRecommendationMIMIC3
from pyhealth.trainer import Trainer
import torch

from google.colab import drive
drive.mount('/content/drive')

if __name__ == "__main__":
    # STEP 1: load data
    base_dataset = MIMIC3Dataset(
        root="/content/drive/MyDrive/mimic-iii-clinical-database-1.4",
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        cache_dir=tempfile.TemporaryDirectory().name,
        dev=True,
    )
    base_dataset.stats()

    # STEP 2: set task
    task = DrugRecommendationMIMIC3()
    sample_dataset = base_dataset.set_task(task)

    train_dataset, val_dataset, test_dataset = split_by_patient(
        sample_dataset, [0.8, 0.1, 0.1]
    )
    train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
    val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
    test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

    # STEP 3: define model
    model = Transformer(
        dataset=sample_dataset,
    )

    # STEP 4: define trainer
    trainer = Trainer(
        model=model,
        metrics=["jaccard_samples", "f1_samples", "pr_auc_samples"],
    )

    trainer.train(
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        epochs=1,
        monitor="pr_auc_samples",
    )

    # STEP 5: evaluate
    print(trainer.evaluate(test_dataloader))

    #precision@k and recall@k metrics
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in test_dataloader:
            output = model(**batch)

            y_true.append(batch["drugs"].cpu())

            y_pred.append(output["y_prob"].cpu())

    y_true = torch.cat(y_true)
    y_pred = torch.cat(y_pred)

    def precision_at_k(y_true, y_pred, k):
      topk = torch.topk(y_pred, k, dim=1).indices

      precisions = []

      for i in range(y_true.shape[0]):
          true_set = set(torch.where(y_true[i] == 1)[0].tolist())
          pred_set = set(topk[i].tolist())

          precisions.append(len(true_set & pred_set) / k)

      return sum(precisions) / len(precisions)

    def recall_at_k(y_true, y_pred, k):
      topk = torch.topk(y_pred, k, dim=1).indices

      recalls = []

      for i in range(y_true.shape[0]):
          true_set = set(torch.where(y_true[i] == 1)[0].tolist())
          pred_set = set(topk[i].tolist())

          if len(true_set) == 0:
              recalls.append(0)
          else:
              recalls.append(len(true_set & pred_set) / len(true_set))

      return sum(recalls) / len(recalls)

    print("Precision@10:", precision_at_k(y_true, y_pred, 10))
    print("Recall@10:", recall_at_k(y_true, y_pred, 10))

    print("Precision@20:", precision_at_k(y_true, y_pred, 20))
    print("Recall@20:", recall_at_k(y_true, y_pred, 20))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
No config path provided, using default config


INFO:pyhealth.datasets.mimic3:No config path provided, using default config


Initializing mimic3 dataset from /content/drive/MyDrive/mimic-iii-clinical-database-1.4 (dev mode: True)


INFO:pyhealth.datasets.base_dataset:Initializing mimic3 dataset from /content/drive/MyDrive/mimic-iii-clinical-database-1.4 (dev mode: True)


Using provided cache_dir: /tmp/tmpq5riwekl/68070818-3390-506f-87b5-c38042ce6436


...

New best pr_auc_samples score (0.2861) at epoch-0, step-8


INFO:pyhealth.trainer:New best pr_auc_samples score (0.2861) at epoch-0, step-8


Loaded best model


INFO:pyhealth.trainer:Loaded best model
Evaluation: 100%|██████████| 2/2 [00:00<00:00, 22.03it/s]


{'jaccard_samples': 0.24209906148045146, 'f1_samples': 0.3805747532930606, 'pr_auc_samples': 0.265919416080545, 'loss': 23.254093170166016}
Precision@10: 0.5318181818181817
Recall@10: 0.15412168507514162
Precision@20: 0.5056818181818182
Recall@20: 0.2904528218072841
